### Structured Output

Pydantic

In [6]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:openai/gpt-oss-120b")

In [7]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")


In [8]:
model_with_pydantic=model.with_structured_output(Movie)
model_with_pydantic

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001BCCE257610>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001BCCE2CC050>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The ti

In [13]:
response=model.invoke("Provide the details about movie KGF")
response

AIMessage(content='**KGF – “Kolar Gold Fields” – Overview & Key Details**\n\n| Item | Information |\n|------|--------------|\n| **Full Title** | **KGF: Chapter\u202f1** (2018)  &  **KGF: Chapter\u202f2** (2022) |\n| **Language** | Primarily Kannada (dubbed in Hindi, Tamil, Telugu, Malayalam, and other Indian languages) |\n| **Genre** | Action‑drama, period‑film, crime‑thriller, masala |\n| **Setting** | The 1970s‑80s Kolar Gold Fields (present‑day Karnataka, India) and the underworld of Bombay/Mumbai |\n| **Director / Writer** | Prashanth Neel |\n| **Producer(s)** | Vijay Kiragandur (Hombale Films) |\n| **Music Composer** | Ravi Basrur |\n| **Cinematography** | Bhuvan Gowda |\n| **Editor** | Ujwal Kulkarni |\n| **Production Company** | Hombale Films |\n| **Distribution** | Various regional distributors; worldwide rights handled by Zee Studios (Hindi) and other partners |\n| **Runtime** | Chapter\u202f1 – 2\u202fh\u202f41\u202fmin (≈161\u202fmin) <br> Chapter\u202f2 – 3\u202fh\u202f03\u

In [12]:
response=model_with_pydantic.invoke("Provide the details about movie KGF")
response

Movie(title='KGF: Chapter 1', year=2018, director='Prashanth Neel', rating=8.3)

### Message  output alongside parsed structure

In [14]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """ A movie with details."""
    title:str=Field(..., description="The title of the movie")
    year:int=Field(..., description="This year the movie was released")
    director:str=Field(..., description="The director of the movie")
    rating:float=Field(..., description="The movies rating out of 10")

model_with_pydantic=model.with_structured_output(Movie,include_raw=True)

response=model_with_pydantic.invoke("Provide the details about movie KGF")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "Provide the details about movie KGF". Need to give details: title, director, rating, year. We don\'t have internal knowledge, but we can use function to fetch details. Use function Movie with parameters. Need director, rating, title, year. Title is "KGF". Director is Prashanth Neel. Year 2018 (KGF: Chapter 1) but maybe they refer to the series. Could give both chapters? Probably just KGF (Chapter 1). Rating? Not sure. Could approximate? Better to fetch via function? The function likely returns data. Let\'s call with title "KGF".', 'tool_calls': [{'id': 'fc_5d18a660-8a58-41df-bb98-ceb70bb0981f', 'function': {'arguments': '{"director":"Prashanth Neel","rating":8.3,"title":"KGF","year":2018}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 187, 'prompt_tokens': 163, 'total_tokens': 350, 'completion_time': 0.392712789, 'completion_tokens_details': {'reasoning

### Nested Structure

In [21]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast: list[Actor]
    genres:list[str]
    budget:float | None =Field(None, description="Budget in lakhs INR")

model_with_pydantic=model.with_structured_output(MovieDetails)

response=model_with_pydantic.invoke("Provide the details about movie KGF")
response

MovieDetails(title='KGF: Chapter 1', year=2018, cast=[Actor(name='Yash', role='Rocky'), Actor(name='Srinidhi Shetty', role='Reena'), Actor(name='Ramachandra Raju', role='Garuda'), Actor(name='Anant Nag', role='Mayor Prabhakar Rao'), Actor(name='Malavika Avinash', role='Shanthamma')], genres=['Action', 'Drama'], budget=80000000.0)

### TypedDict

In [32]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """ A movie with details."""
    title:Annotated[str, ...,"The title of the movie"]
    year:Annotated[int, ..., "This year the movie was released"]
    director:Annotated[str, ..., "The director of the movie"]
    rating:Annotated[float, ..., "The movies rating out of 10"]

model_with_typeddict=model.with_structured_output(MovieDict)

response=model_with_typeddict.invoke("Provide the details about movie KGF")
response

{'director': 'Prashanth Neel',
 'rating': 8.3,
 'title': 'KGF: Chapter 1',
 'year': 2018}

In [37]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast: list[Actor]
    genres:list[str]
    budget:float | None =Field(None, description="Budget in lakhs INR")

model_with=model.with_structured_output(MovieDetails,method="json_mode")

response = model_with.invoke(
    "Provide the details about movie KGF in JSON format"  # <-- Added 'in JSON format'
)
response

{'Title': 'KGF: Chapter 1',
 'ReleaseYear': 2018,
 'Director': 'Prashanth Neel',
 'Producer': 'Vijay Kiragandur',
 'Cast': ['Yash',
  'Sanjay Dutt',
  'Raveena Tandon',
  'Ramachandra Raju',
  'Anant Nag'],
 'Language': 'Kannada',
 'RuntimeMinutes': 165,
 'Genre': ['Action', 'Drama'],
 'Plot': 'The story follows Rocky, a young man who rises from poverty to become a powerful gangster in the Kolar Gold Fields, seeking wealth and power.',
 'BoxOffice': {'India': '₹250 crore', 'Worldwide': '₹500 crore'},
 'MusicBy': 'Ravi Basrur',
 'Sequel': 'KGF: Chapter 2 (2022)'}

In [40]:
# model_with_typeddict.profile   #will give error has structured output don't have profile
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}